# 🚚 Delivery ETA Predictor — EDA & Modeling Notebook

This notebook covers:
1. Dataset overview & distribution analysis
2. Geospatial visualization
3. Feature correlation analysis
4. Model training & evaluation
5. Feature importance & interpretation

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

from data.generate_data import generate_dataset
from src.feature_engineering import build_features
from config import FEATURE_CONFIG

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK ✓')

## 1. Generate & Load Dataset

In [ ]:
# Generate synthetic dataset (skip if already exists)
import os
if not os.path.exists('../data/raw_data.csv'):
    generate_dataset()

raw_df = pd.read_csv('../data/raw_data.csv')
print(f'Dataset shape: {raw_df.shape}')
raw_df.head()

In [ ]:
# Dataset statistics
raw_df.describe().round(2)

## 2. Target Distribution — Delivery Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(raw_df['delivery_time_minutes'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Delivery Time Distribution', fontweight='bold')
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Count')
axes[0].axvline(raw_df['delivery_time_minutes'].mean(), color='red', linestyle='--', label=f'Mean: {raw_df["delivery_time_minutes"].mean():.1f} min')
axes[0].legend()

raw_df.boxplot(column='delivery_time_minutes', by='traffic_level', ax=axes[1])
axes[1].set_title('Delivery Time by Traffic Level', fontweight='bold')
axes[1].set_xlabel('Traffic Level')
axes[1].set_ylabel('Minutes')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
featured_df = build_features(raw_df, training=True)
print(f'Engineered features shape: {featured_df.shape}')
featured_df.head()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
corr = featured_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 4. Temporal Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly_avg = featured_df.groupby('hour_of_day')['delivery_time_minutes'].mean()
axes[0].plot(hourly_avg.index, hourly_avg.values, marker='o', color='steelblue', linewidth=2)
axes[0].fill_between(hourly_avg.index, hourly_avg.values, alpha=0.2, color='steelblue')
axes[0].axvspan(8, 10, alpha=0.15, color='red', label='Morning peak')
axes[0].axvspan(17, 20, alpha=0.15, color='orange', label='Evening peak')
axes[0].set_title('Avg Delivery Time by Hour', fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg ETA (min)')
axes[0].legend()

day_avg = featured_df.groupby('day_of_week')['delivery_time_minutes'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors = ['#e74c3c' if d >= 5 else '#3498db' for d in range(7)]
axes[1].bar(days, day_avg.values, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Avg Delivery Time by Day', fontweight='bold')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Avg ETA (min)')

plt.tight_layout()
plt.show()

## 5. Model Training & Evaluation

In [ ]:
from src.model_training import run_training
xgb_model, rf_model, scaler, metadata = run_training()

In [ ]:
# Feature importance — XGBoost
import pandas as pd

feat_names = metadata['feature_columns']
importances = pd.Series(xgb_model.feature_importances_, index=feat_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('XGBoost Feature Importances', fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 6. Prediction Examples

In [ ]:
from src.predict import predict_single

examples = [
    {'pickup_lat': 12.9716, 'pickup_lon': 77.5946, 'drop_lat': 13.0012, 'drop_lon': 77.6141,
     'traffic_level': 'low', 'order_timestamp': '2024-03-15 14:00:00'},
    {'pickup_lat': 12.9716, 'pickup_lon': 77.5946, 'drop_lat': 13.0012, 'drop_lon': 77.6141,
     'traffic_level': 'high', 'order_timestamp': '2024-03-15 18:30:00'},
    {'pickup_lat': 12.8500, 'pickup_lon': 77.4600, 'drop_lat': 13.0800, 'drop_lon': 77.7000,
     'traffic_level': 'very_high', 'order_timestamp': '2024-03-15 18:00:00'},
]

for i, ex in enumerate(examples, 1):
    result = predict_single(ex)
    print(f'Example {i} | Traffic: {ex["traffic_level"]:10} | Distance: {result["distance_km"]} km | ETA: {result["predicted_eta_minutes"]} min')